# Fine-tuning de BERT sur IMDB

Fine-tuning de **`bert-base-uncased`** avec Hugging Face `Trainer` pour la classification de sentiment sur **50 000 avis
IMDB**, puis comparaison avec la baseline BiLSTM / BiGRU (`01_lstm_gru_baseline.ipynb`).

**Principes de la comparaison**
- Même découpage 80 / 10 / 10 que le notebook LSTM/GRU (même graine ; l'empreinte du jeu de test doit être identique).
- Le meilleur point de contrôle est choisi sur la **validation** ; le jeu de test n'est évalué qu'une fois, à la fin.
- Les avis sont tronqués à `MAX_LENGTH` jetons : c'est un compromis vitesse / précision (la médiane d'un avis est d'environ 175 mots).

**Exécution :** Colab → *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Tout exécuter*. Durée : environ 30 à 40 minutes.

In [1]:
!pip install -q transformers datasets accelerate scikit-learn matplotlib

In [2]:
import hashlib
import inspect
import json
import os
import platform
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed as hf_set_seed)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
hf_set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (très lent : activer un GPU)")

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

Device : cuda | Tesla T4


## 1. Données (même jeu et même découpage que la baseline)

In [3]:
from datasets import load_dataset


def load_imdb():
    """IMDB : 25 000 avis d'entraînement + 25 000 de test = 50 000 avis équilibrés."""
    ds = load_dataset("stanfordnlp/imdb")
    df = pd.concat([ds["train"].to_pandas(), ds["test"].to_pandas()], ignore_index=True)
    df = df.rename(columns={"text": "review"})
    return df[["review", "label"]]  # label : 0 = négatif, 1 = positif


df = load_imdb()
print("Nombre d'avis :", len(df))
print(df["label"].value_counts().rename({0: "négatifs", 1: "positifs"}))

Nombre d'avis : 50000
label
négatifs    25000
positifs    25000
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
import hashlib

# Découpage stratifié 80 / 10 / 10. IDENTIQUE dans les deux notebooks (même graine, même code).
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED)

# Empreinte du jeu de test : doit être identique dans les deux notebooks.
SPLIT_FINGERPRINT = hashlib.md5("".join(test_df["review"].tolist()).encode("utf-8")).hexdigest()[:12]
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Empreinte du jeu de test :", SPLIT_FINGERPRINT)

Train: 40000 | Val: 5000 | Test: 5000
Empreinte du jeu de test : f158b26f6f35


## 2. Tokenisation

Tokenizer WordPiece de `bert-base-uncased`, troncature à `MAX_LENGTH` jetons, rembourrage dynamique par lot.

In [5]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def clean(text):
    return re.sub(r"<br\s*/?>", " ", text)  # balises HTML présentes dans IMDB


def to_dataset(frame):
    ds = Dataset.from_pandas(
        frame.assign(text=frame["review"].map(clean))[["text", "label"]].rename(columns={"label": "labels"}),
        preserve_index=False,
    )
    return ds.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
                  batched=True, remove_columns=["text"])


train_ds, val_ds, test_ds = to_dataset(train_df), to_dataset(val_df), to_dataset(test_df)
token_lengths = np.array([len(ids) for ids in train_ds["input_ids"]])
print(f"Avis tronqués à {MAX_LENGTH} jetons : {(token_lengths >= MAX_LENGTH).mean():.1%}")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Avis tronqués à 256 jetons : 40.8%


## 3. Fine-tuning

`bert-base-uncased` + tête de classification (2 classes). 2 époques, taux d'apprentissage 2e-5, warmup 10 %, weight decay 0.01,
précision mixte (fp16) sur GPU. Le meilleur point de contrôle (accuracy de validation) est rechargé à la fin.

In [6]:
EPOCHS, BATCH_SIZE, LEARNING_RATE = 2, 16, 2e-5

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds), "precision": precision, "recall": recall, "f1": f1}


# Warmup de 10 % des pas : le nom du paramètre dépend de la version de transformers.
warmup = {"warmup_ratio": 0.1} if "warmup_ratio" in inspect.signature(TrainingArguments).parameters else {"warmup_steps": 0.1}

args = TrainingArguments(
    output_dir="bert_imdb",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    **warmup,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
train_time = time.time() - start
print(f"\nTemps d'entraînement : {train_time / 60:.1f} min")

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.232697,0.224585,0.921600,0.923633,0.919200,0.921411
2,0.167210,0.268380,0.925400,0.926937,0.923600,0.925265


[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte


Temps d'entraînement : 17.7 min


## 4. Évaluation sur le jeu de test

In [7]:
val_metrics = trainer.evaluate(val_ds)
pred_output = trainer.predict(test_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = logits.argmax(axis=-1)
probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
test_metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
    "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
}
(tn, fp), (fn, tp) = test_metrics["confusion_matrix"]
print(f"Validation (meilleur point de contrôle) : accuracy {val_metrics['eval_accuracy']:.4f}")
print(f"TEST : accuracy {test_metrics['accuracy']:.4f} | précision {precision:.4f} | rappel {recall:.4f} | F1 {f1:.4f}")
print(f"Vrais négatifs {tn} | faux positifs {fp} | faux négatifs {fn} | vrais positifs {tp}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.167210,0.268380,2,0.925400,0.926937,0.923600,0.925265


Validation (meilleur point de contrôle) : accuracy 0.9254
TEST : accuracy 0.9308 | précision 0.9284 | rappel 0.9336 | F1 0.9310
Vrais négatifs 2320 | faux positifs 180 | faux négatifs 166 | vrais positifs 2334


In [ ]:
history = trainer.state.log_history
train_points = [(h["epoch"], h["loss"]) for h in history if "loss" in h and "eval_loss" not in h]
eval_points = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]
plt.figure(figsize=(7, 4.5))
plt.plot(*zip(*train_points), label="perte d'entraînement", alpha=0.7)
plt.plot(*zip(*eval_points), "o-", label="perte de validation")
plt.xlabel("Époque")
plt.ylabel("Perte")
plt.title("Fine-tuning de BERT")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "bert_curves.png"), dpi=130)
plt.show()

## 5. Analyse des erreurs

In [9]:
errors = test_df.assign(pred=y_pred, proba_positive=probs)
errors = errors[errors["pred"] != errors["label"]]
print(f"{len(errors)} erreurs sur {len(test_df)} avis de test\n")
for _, row in errors.sample(6, random_state=SEED).iterrows():
    truth = "positif" if row["label"] == 1 else "négatif"
    print(f"[vrai : {truth} | P(positif) = {row['proba_positive']:.2f}] {clean(row['review'])[:260]}...\n")

346 erreurs sur 5000 avis de test

[vrai : négatif | P(positif) = 0.76] Audrey, I know you truly cherish your husband Ted's memory but PLEASE do his legacy justice and heed his wishes. Dr. Seuss refused to license his characters during his lifetime for a very good reason. We beg of you to please stop cashing in on his stories, ima...

[vrai : négatif | P(positif) = 0.60] i wrote an essay in 1981, the year i graduated high school called the "last American virgin." i also had a friend named nancy who was the prostitute in this film. apparently her daughter got a hold of my essay one night when she slept over my house. a year lat...

[vrai : positif | P(positif) = 0.10] Matthau is a widowed hospital doctor enjoying his single status and the footloose and available nurses on the staff whilst colleague and friend Richard Benjamin looks on with amusement and amazement. Their boss is hard-of-hearing going on senile Chief of Staff...

[vrai : négatif | P(positif) = 0.91] I'm not sure quite why

In [10]:
def predict_sentiment(text):
    inputs = tokenizer(clean(text), truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        p = torch.softmax(trainer.model(**inputs).logits, dim=-1)[0, 1].item()
    return ("positif" if p > 0.5 else "négatif"), p


for sentence in ["This movie was an absolute masterpiece. I loved every second!",
                 "A boring, predictable mess. I wanted my money back."]:
    label, p = predict_sentiment(sentence)
    print(f"{label} ({p:.3f}) <- {sentence}")

positif (0.998) <- This movie was an absolute masterpiece. I loved every second!
négatif (0.008) <- A boring, predictable mess. I wanted my money back.


## 6. Sauvegarde des résultats

In [11]:
payload = {
    "task": "IMDB sentiment, fine-tuning de bert-base-uncased",
    "split": {"train": len(train_df), "val": len(val_df), "test": len(test_df), "seed": SEED,
              "test_fingerprint": SPLIT_FINGERPRINT},
    "config": {"model": MODEL_NAME, "max_length": MAX_LENGTH, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
               "learning_rate": LEARNING_RATE, "weight_decay": 0.01, "warmup": 0.1, "fp16": torch.cuda.is_available()},
    "environment": {"device": str(device), "torch": torch.__version__, "transformers": transformers.__version__,
                    "python": platform.python_version(),
                    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    "validation_accuracy": float(val_metrics["eval_accuracy"]),
    "test": test_metrics,
    "train_time_s": round(train_time, 1),
    "n_parameters": sum(p.numel() for p in trainer.model.parameters()),
    "log_history": history,
}
out_path = os.path.join(RESULTS_DIR, "bert_metrics.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print("Résultats enregistrés :", out_path)

try:  # téléchargement automatique dans Colab
    from google.colab import files
    files.download(out_path)
    files.download(os.path.join(RESULTS_DIR, "bert_curves.png"))
except ImportError:
    pass

Résultats enregistrés : results/bert_metrics.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>